Installing

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn
!pip install pandas
!pip install tqdm
!pip install scikit-image
!pip install scipy
!pip install ace_tools

Libraries

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
import glob
import os

Train

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize

# === CONFIGURATION ===
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

IMAGE_SIZE = (100, 300)
PATCH_SIZE = 10
stride = 10
NUM_SUBJECTS = 123
NUM_FINGERS = 4
TRAIN_INDICES = [1, 2, 3]
LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]
NUM_ROW_COMPONENTS = 137
NUM_COL_COMPONENTS = 137

# === MAPPING ===
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        table[i] = sum(min_rotation) if transitions <= 2 else P + 1
    return table

mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# === LBP HISTOGRAM ===
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# === FEATURE EXTRACTION ===
train_lbp_images = []
train_labels = []

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Subjects"):
    for finger_id in range(1, NUM_FINGERS + 1):
        for img_idx in TRAIN_INDICES:
            for session_path in [base_path_sess1, base_path_sess2]:
                session_name = "1st" if "1st" in session_path else "2nd"
                folder = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(session_path, folder, f"{img_idx:02d}.jpg")
                print(f"📥 Reading: {img_path}")

                if not os.path.exists(img_path):
                    print(f"❌ Missing file: {img_path}")
                    continue

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Cannot open image: {img_path}")
                    continue

                img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
                img = cv2.fastNlMeansDenoising(img, h=10)
                img = cv2.equalizeHist(img)

                h_patches = (img.shape[0] - PATCH_SIZE) // stride + 1
                w_patches = (img.shape[1] - PATCH_SIZE) // stride + 1

                lbp_matrix = np.zeros((h_patches, w_patches * sum(P + 2 for _, P in LBP_CONFIGS)))

                for i, y in enumerate(range(0, img.shape[0] - PATCH_SIZE + 1, stride)):
                    row_features = []
                    for x in range(0, img.shape[1] - PATCH_SIZE + 1, stride):
                        block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        if block.shape != (PATCH_SIZE, PATCH_SIZE):
                            continue
                        block_hist = []
                        for R, P in LBP_CONFIGS:
                            hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            block_hist.extend(hist)
                        row_features.append(block_hist)
                    if row_features:
                        lbp_matrix[i, :] = np.hstack(row_features)

                train_lbp_images.append(lbp_matrix)
                label = f"{subject_id:03d}_f{finger_id}_img{img_idx:02d}_{session_name}"
                train_labels.append(label)
                print(f"✅ Features extracted for {label}")

# === VALIDATE ===
if len(train_lbp_images) == 0:
    print("❌ No training data extracted. Please check the image paths and TRAIN_INDICES.")
else:
    print(f"\n📊 Total training samples: {len(train_lbp_images)}")

# === (2D)^2PCA TRAINING ===
def compute_2d2pca_projection(images, num_row_components, num_col_components):
    print("\n⚙️ Computing (2D)²PCA projection matrices...")
    n = len(images)
    h, w = images[0].shape
    mean_img = sum(images) / n
    G_row = np.zeros((h, h))
    G_col = np.zeros((w, w))
    for A in images:
        A = A - mean_img
        G_row += A @ A.T
        G_col += A.T @ A
    G_row /= n
    G_col /= n
    eig_vals_r, eig_vecs_r = np.linalg.eigh(G_row)
    eig_vals_c, eig_vecs_c = np.linalg.eigh(G_col)
    U = eig_vecs_r[:, np.argsort(-eig_vals_r)[:num_row_components]]
    V = eig_vecs_c[:, np.argsort(-eig_vals_c)[:num_col_components]]
    return U, V

# === COMPUTE U, V ===
U, V = compute_2d2pca_projection(train_lbp_images, NUM_ROW_COMPONENTS, NUM_COL_COMPONENTS)

# === PROJECT IMAGES ===
projected_features = [U.T @ A @ V for A in train_lbp_images]
flat_features = np.array([f.flatten() for f in projected_features])
train_labels = np.array(train_labels)

print("\n✅ Training complete.")
print("📐 Final shape:", flat_features.shape)
print("🧾 Sample labels:", train_labels[:5])


Test

In [ ]:
# Modified TEST feature extraction for FV-USM Protocol 1 Strategy 2 using (2D)^2PCA

import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern

IMAGE_SIZE = (100, 300)
PATCH_SIZE = 10
stride = 10
NUM_SUBJECTS = 123
NUM_FINGERS = 4
TEST_INDICES = [4, 5, 6]
LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]

# === RIU2 MAPPING ===
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        table[i] = sum(min_rotation) if transitions <= 2 else P + 1
    return table

mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# === LBP HISTOGRAM ===
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# === FEATURE EXTRACTION ===
test_lbp_images = []
test_labels = []

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Subjects"):
    for finger_id in range(1, NUM_FINGERS + 1):
        for img_idx in TEST_INDICES:
            for session_path in [base_path_sess1, base_path_sess2]:
                session_name = "1st" if "1st" in session_path else "2nd"
                folder = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(session_path, folder, f"{img_idx:02d}.jpg")
                print(f"📥 Reading: {img_path}")

                if not os.path.exists(img_path):
                    print(f"❌ Missing file: {img_path}")
                    continue

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Cannot open image: {img_path}")
                    continue

                img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
                img = cv2.fastNlMeansDenoising(img, h=10)
                img = cv2.equalizeHist(img)

                h_patches = (img.shape[0] - PATCH_SIZE) // stride + 1
                w_patches = (img.shape[1] - PATCH_SIZE) // stride + 1

                lbp_matrix = np.zeros((h_patches, w_patches * sum(P + 2 for _, P in LBP_CONFIGS)))

                for i, y in enumerate(range(0, img.shape[0] - PATCH_SIZE + 1, stride)):
                    row_features = []
                    for x in range(0, img.shape[1] - PATCH_SIZE + 1, stride):
                        block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        if block.shape != (PATCH_SIZE, PATCH_SIZE):
                            continue
                        block_hist = []
                        for R, P in LBP_CONFIGS:
                            hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            block_hist.extend(hist)
                        row_features.append(block_hist)
                    if row_features:
                        lbp_matrix[i, :] = np.hstack(row_features)

                test_lbp_images.append(lbp_matrix)
                label = f"{subject_id:03d}_f{finger_id}_img{img_idx:02d}_{session_name}"
                test_labels.append(label)
                print(f"✅ Features extracted for {label}")

# === PROJECT USING (2D)^2PCA ===
try:
    test_projected = [U.T @ A @ V for A in test_lbp_images]
    test_flat_features = np.array([f.flatten() for f in test_projected])
    test_labels = np.array(test_labels)

    print("\n✅ Test projection completed.")
    print("📐 Projected feature shape:", test_flat_features.shape)
    print("🧾 Sample labels:", test_labels[:5])

except NameError:
    print("❌ Error: U and V projection matrices from training are not defined. Run the training code first.")



Session Sensitive

In [ ]:
# === SESSION-SENSITIVE EVALUATION FOR FV-USM S2 + (2D)²PCA ===
correct_matches = 0
total_tests = len(test_labels)

print("\n🔍 Step 1: Starting Session-Sensitive Evaluation using Manhattan distance in (2D)²PCA space...\n")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    true_label = test_labels[i]  # e.g., '123_f2_img04_1st'

    # 📏 Manhattan distance to all training vectors
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]  # e.g., '123_f2_img03_1st'

    # 🎯 Extract key label parts: subject ID, finger ID, session
    true_parts = true_label.split('_')        # ['123', 'f2', 'img04', '1st']
    pred_parts = predicted_label.split('_')   # ['123', 'f2', 'img03', '1st']

    true_subj, true_finger, true_session = true_parts[0], true_parts[1], true_parts[-1]
    pred_subj, pred_finger, pred_session = pred_parts[0], pred_parts[1], pred_parts[-1]

    if (true_subj == pred_subj) and (true_finger == pred_finger) and (true_session == pred_session):
        correct_matches += 1
        result = "✅ CORRECT"
        emoji = "🎯"
    else:
        result = "❌ WRONG"
        emoji = "⚠️"

    print(f"{emoji} Test sample {i+1}/{total_tests}")
    print(f"    🔍 Predicted label: {predicted_label}")
    print(f"    🎯 Ground Truth   : {true_label}")
    print(f"    ➡️  Match Result  : {result}\n")

# === FINAL ACCURACY ===
accuracy = (correct_matches / total_tests) * 100
print("📊 Final Results: Session-Sensitive Evaluation for Strategy 2 (Finger-wise)")
print(f"✅ Correct Matches : {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy: {accuracy:.2f}%\n")


Session Independent

In [ ]:
# === SESSION-INDEPENDENT EVALUATION for FV-USM S2 + (2D)^2PCA ===
correct_matches = 0
total_tests = len(test_labels)

print("\n🔍 Step 1: Starting Session-Independent Evaluation using Manhattan distance in (2D)²PCA space...\n")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    true_label = test_labels[i]  # e.g., '007_f2_img06_2nd'

    # 📏 Manhattan distance to all training samples
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]  # e.g., '007_f2_img02_1st'

    # 🎯 Compare subject ID and finger ID (ignore session)
    true_id = "_".join(true_label.split('_')[:2])      # '007_f2'
    pred_id = "_".join(predicted_label.split('_')[:2]) # '007_f2'

    if pred_id == true_id:
        correct_matches += 1
        result = "✅ CORRECT"
        emoji = "🎯"
    else:
        result = "❌ WRONG"
        emoji = "⚠️"

    print(f"{emoji} Test sample {i+1}/{total_tests}")
    print(f"    🧾 Predicted: {predicted_label}")
    print(f"    🎯 Actual   : {true_label}")
    print(f"    ➡️  Result   : {result}\n")

# === FINAL ACCURACY ===
accuracy = (correct_matches / total_tests) * 100
print("📊 Final Results (Session-Independent Evaluation for Strategy 2 - Finger-Wise)")
print(f"✅ Correct Matches : {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy: {accuracy:.2f}%\n")


Session Sensitive R5

In [ ]:
import numpy as np
from collections import defaultdict

ranks = [1, 5]
rank_correct = defaultdict(int)
total_tests = len(test_labels)

print("📊 Calculating Session-Sensitive CMC (Rank-1 & Rank-5)...")

for i in range(total_tests):
    test_label = test_labels[i]
    proj_test = test_flat_features[i]

    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_finger = test_parts[1]
    test_session = test_parts[-1]
    test_id = f"{test_subject}_{test_finger}_{test_session}"

    # Use all training samples — no filtering!
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    matched = False
    for r in range(1, max(ranks) + 1):
        candidate_label = train_labels[sorted_indices[r - 1]]
        parts = candidate_label.split('_')
        candidate_subject = parts[0]
        candidate_finger = parts[1]
        candidate_session = parts[-1]
        candidate_id = f"{candidate_subject}_{candidate_finger}_{candidate_session}"

        # ✅ Match only if subject, finger, and session all match
        if candidate_id == test_id and not matched:
            for k in ranks:
                if r <= k:
                    rank_correct[k] += 1
            matched = True

# === FINAL RESULTS ===
for k in ranks:
    accuracy = (rank_correct[k] / total_tests) * 100
    print(f"🎯 Rank-{k} Accuracy (Subject + Finger + Session): {accuracy:.2f}%")


Session Sensitive CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
total_tests = len(test_labels)

print("📊 Calculating Session-Sensitive CMC Curve (Rank-1 to Rank-100)...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    test_parts = test_labels[i].split('_')
    true_subject = test_parts[0]
    true_finger = test_parts[1]
    true_session = test_parts[-1]
    true_id = f"{true_subject}_{true_finger}_{true_session}"

    # Compute Manhattan distances to all training samples
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # Search for the first correct match (full subject + finger + session match)
    for r in range(max_rank):
        candidate_label = train_labels[sorted_indices[r]]
        cand_parts = candidate_label.split('_')
        cand_subject = cand_parts[0]
        cand_finger = cand_parts[1]
        cand_session = cand_parts[-1]
        candidate_id = f"{cand_subject}_{cand_finger}_{cand_session}"

        if candidate_id == true_id:
            rank_correct[r:] += 1  # all ranks >= r count as correct
            break

# === Normalize to percentage
cmc_curve = (rank_correct / total_tests) * 100

# === Plotting the CMC Curve
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Sensitive CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.title("Session-Sensitive CMC Curve — LBP$_{\\mathrm{RIU2}}$((8,1), (16,1), (8,2)) + (2D)$^2$PCA (Strategy 2, Protocol 2)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank+1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# === Print key ranks
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Sensitive Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

print("\n📊 Evaluating (Session-Sensitive): LBP((8,1),(16,1),(8,2)) + (2D)^2PCA over all test-train pairs...\n")

all_scores = []
all_labels = []

for test_idx in range(len(test_flat_features)):
    test_vec = test_flat_features[test_idx]
    test_label = test_labels[test_idx]
    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_finger = test_parts[1]
    test_session = test_parts[-1]
    test_id = f"{test_subject}_{test_finger}_{test_session}"

    for train_idx in range(len(flat_features)):
        train_vec = flat_features[train_idx]
        train_label = train_labels[train_idx]
        train_parts = train_label.split('_')
        train_subject = train_parts[0]
        train_finger = train_parts[1]
        train_session = train_parts[-1]
        train_id = f"{train_subject}_{train_finger}_{train_session}"

        # ✅ Negative Manhattan distance (higher = more similar)
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # ✅ Genuine if subject, finger, and session match
        is_genuine = int(test_id == train_id)
        all_labels.append(is_genuine)

# === Normalize scores to [0, 1]
scores = np.array(all_scores)
labels = np.array(all_labels)
scores = (scores - scores.min()) / (scores.max() - scores.min())

# === Threshold Sweeping for best F1
best_f1 = best_thresh = best_prec = best_rec = 0
for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = precision
        best_rec = recall

# === Final results at best threshold
final_preds = (scores >= best_thresh).astype(int)
accuracy = accuracy_score(labels, final_preds)

# === Output summary for your paper/table
print("🔍 Summary (Session-Sensitive): LBP((8,1),(16,1),(8,2)) + (2D)^2PCA")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {accuracy * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")


Session Independent R5

In [ ]:
import heapq
from collections import defaultdict

# === CMC Calculation ===
ranks = [1, 5]  # for Rank-1 and Rank-5
rank_correct = defaultdict(int)
total_tests = len(test_labels)

print("📊 Calculating CMC, Rank-1, and Rank-5...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    true_id = "_".join(test_labels[i].split('_')[:2])

    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    top_k_indices = np.argsort(distances)

    found = False
    for r in range(1, max(ranks)+1):
        candidate_label = train_labels[top_k_indices[r-1]]
        candidate_id = "_".join(candidate_label.split('_')[:2])
        if candidate_id == true_id and not found:
            for k in ranks:
                if r <= k:
                    rank_correct[k] += 1
            found = True

# === Print Results ===
for k in ranks:
    cmc_score = (rank_correct[k] / total_tests) * 100
    print(f"🎯 Rank-{k} Accuracy: {cmc_score:.2f}%")


Session Independent CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# === CONFIGURATION ===
max_rank = 100  # Now from Rank-1 to Rank-100
rank_correct = np.zeros(max_rank)
total_tests = len(test_labels)

print("📊 Calculating CMC curve (Rank-1 to Rank-100)...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    true_id = "_".join(test_labels[i].split('_')[:2])

    # Compute Manhattan distances and sort them
    distances = np.sum(np.abs(flat_features - proj_test), axis=1)
    top_k_indices = np.argsort(distances)

    # Search for the first correct match and update CMC counts
    for r in range(max_rank):
        candidate_label = train_labels[top_k_indices[r]]
        candidate_id = "_".join(candidate_label.split('_')[:2])
        if candidate_id == true_id:
            rank_correct[r:] += 1
            break  # Found the correct match, fill from r to end

# === Normalize to percentage accuracy
cmc_curve = (rank_correct / total_tests) * 100

# === Plotting
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="CMC Curve", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Accuracy (%)")
plt.title("Session-Independent CMC Curve — LBP$_{\\mathrm{RIU2}}$((8,1), (16,1), (8,2)) + (2D)$^2$PCA (Strategy 2, Protocol 2)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank+1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# === Print specific ranks
print(f"🎯 Rank-1 Accuracy: {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy: {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy: {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy: {cmc_curve[99]:.2f}%")


Session Independent Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

print("\n📊 Evaluating: LBP((8,1),(16,1),(8,2)) + (2D)^2PCA — Performance over all test-train pairs...\n")

all_scores = []
all_labels = []

for test_idx in range(len(test_flat_features)):
    test_vec = test_flat_features[test_idx]
    test_label = test_labels[test_idx]
    test_id = "_".join(test_label.split('_')[:2])  # e.g., '007_f2'

    for train_idx in range(len(flat_features)):
        train_vec = flat_features[train_idx]
        train_label = train_labels[train_idx]
        train_id = "_".join(train_label.split('_')[:2])  # e.g., '007_f2'

        # Similarity score using negative Manhattan distance (higher is more similar)
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # Ground truth: 1 = genuine match, 0 = impostor
        is_genuine = int(test_id == train_id)
        all_labels.append(is_genuine)

# === Normalize scores to [0, 1]
scores = np.array(all_scores)
labels = np.array(all_labels)
scores = (scores - scores.min()) / (scores.max() - scores.min())

# === Find optimal threshold for maximum F1 score
best_f1 = best_thresh = best_prec = best_rec = 0
for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = precision
        best_rec = recall

# === Final performance at best threshold
final_preds = (scores >= best_thresh).astype(int)
accuracy = accuracy_score(labels, final_preds)

# === Output summary for your table/report
print("🔍 Summary: LBP((8,1),(16,1),(8,2)) + (2D)^2PCA")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {accuracy * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")
